Create Drug Descriptions Embeddings per Visit

In [1]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)


Torch version: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA A10-24Q
VRAM (GB): 25.769345024


In [2]:
import os
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from collections import defaultdict
import xml.etree.ElementTree as ET

# ------------------------------
# CONFIG
# ------------------------------
CHUNK_SIZE = 10_000
INITIAL_BATCH_SIZE = 8
MODEL_NAME = "google/medgemma-4b-pt"
MAX_LEN = 512

# Paths
BRIDGE_PATH = r"....drugbank_mimic_bridge_clean.h5"  # maps MIMIC drugs to DrugBank_ID
PRESCRIPTIONS_PATH = r".......mimic iv\mimic-iv-3.1\hosp\prescriptions.csv.gz"
DRUGBANK_XML_PATH = r"....full database.xml"
DIAGNOSES_PATH = r".....mimic iv\mimic-iv-3.1\hosp\diagnoses_icd.csv.gz"
OUTPUT_PATH = r"......cancer_admission_embs_drugs.npy"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# ------------------------------
# TARGET ICD CODES (SOLID CANCERS)
# ------------------------------
TARGET_ICD9_PREFIXES = (
    "140","141","142","143","144","145","146","147","148","149",
    "153","154","162","174","185","188"
)
TARGET_ICD10_PREFIXES = (
    "C00","C01","C02","C03","C04","C05","C06","C07","C08",
    "C18","C19","C20","C34","C50","C61","C67"
)

# ------------------------------
# DEVICE & MODEL
# ------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModel.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
model.eval()
print("MedGemma model loaded.\n")

# ------------------------------
# EMBEDDING FUNCTIONS
# ------------------------------
@torch.inference_mode()
def embed_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LEN
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model(**inputs)
    attention_mask = inputs["attention_mask"].unsqueeze(-1)
    hidden = outputs.last_hidden_state
    emb = (hidden * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return emb.float().cpu().numpy()

def safe_embed_batch(texts):
    batch = texts
    while len(batch) > 0:
        try:
            return embed_batch(batch)
        except RuntimeError as e:
            if "out of memory" in str(e):
                torch.cuda.empty_cache()
                batch = batch[: len(batch) // 2]
                print(f"OOM → reducing batch to {len(batch)}")
            else:
                raise
    return None

# ------------------------------
# LOAD DIAGNOSES TO FILTER TARGET HADM_IDS
# ------------------------------
diag = pd.read_csv(DIAGNOSES_PATH, usecols=["subject_id","hadm_id","icd_code","icd_version"])
diag["icd_code"] = diag["icd_code"].astype(str).str.upper().str.strip()
target_mask = (
    ((diag.icd_version==9) & diag.icd_code.str.startswith(TARGET_ICD9_PREFIXES)) |
    ((diag.icd_version==10) & diag.icd_code.str.startswith(TARGET_ICD10_PREFIXES))
)
target_diag = diag[target_mask]
target_hadm_ids = set(target_diag["hadm_id"].unique())
print(f"Found {len(target_hadm_ids)} cancer admissions.\n")

# ------------------------------
# LOAD PRESCRIPTIONS
# ------------------------------
prescriptions = pd.read_csv(PRESCRIPTIONS_PATH, usecols=["subject_id","hadm_id","drug"])
prescriptions["drug"] = prescriptions["drug"].astype(str).str.upper().str.strip()
prescriptions = prescriptions[prescriptions["hadm_id"].isin(target_hadm_ids)]
print(f"Filtered prescriptions to {len(prescriptions)} rows for target cancer admissions.\n")

# ------------------------------
# PARSE DRUGBANK XML
# ------------------------------
print("Parsing DrugBank XML...")
tree = ET.parse(DRUGBANK_XML_PATH)
root = tree.getroot()
ns = {"db": "http://www.drugbank.ca"}

drug_desc_map = {}
for drug in root.findall("db:drug", ns):
    db_id = drug.find("db:drugbank-id[@primary='true']", ns)
    if db_id is None: continue
    db_id = db_id.text
    desc = drug.findtext("db:description", default="", namespaces=ns)
    if len(desc) > 5:
        drug_desc_map[db_id.upper()] = desc.strip()

print(f"Total drugs with descriptions parsed: {len(drug_desc_map)}\n")

# ------------------------------
# LOAD BRIDGE (drug → DrugBank_ID)
# ------------------------------
bridge = pd.read_hdf(BRIDGE_PATH, key="bridge")
bridge["drug_upper"] = bridge["drug"].astype(str).str.lower().str.strip()
prescriptions["drug_upper"] = prescriptions["drug"].astype(str).str.lower().str.strip()

# Merge prescriptions to bridge to assign DrugBank_ID
bridge_presc = bridge.merge(
    prescriptions[["hadm_id","subject_id","drug_upper"]],
    left_on="drug_upper",
    right_on="drug_upper",
    how="inner"
)

# ------------------------------
# AGGREGATE DRUG DESCRIPTIONS PER ADMISSION
# ------------------------------
admission_drugs = defaultdict(list)
admission_subject = {}

for _, row in bridge_presc.iterrows():
    hadm_id = row["hadm_id"]
    subj_id = row["subject_id"]
    db_id = row["DrugBank_ID"]
    desc = drug_desc_map.get(db_id, None)
    if desc is None:  # skip if no description in XML
        continue
    admission_drugs[hadm_id].append(desc)
    admission_subject[hadm_id] = subj_id

print(f"Total admissions with drug descriptions: {len(admission_drugs)}\n")

# ------------------------------
# COMPUTE ADMISSION-LEVEL EMBEDDINGS
# ------------------------------
admission_embeddings = []
admission_ids = []
subject_ids = []

for i, (hid, descs) in enumerate(admission_drugs.items(), start=1):
    embs = []
    for j in range(0, len(descs), INITIAL_BATCH_SIZE):
        batch = descs[j : j + INITIAL_BATCH_SIZE]
        batch_emb = safe_embed_batch(batch)
        if batch_emb is not None:
            embs.append(batch_emb)
    if not embs:
        continue
    admission_emb = np.vstack(embs).mean(axis=0)
    if np.isnan(admission_emb).any():
        continue
    admission_embeddings.append(admission_emb)
    admission_ids.append(hid)
    subject_ids.append(admission_subject[hid])
    if i % 50 == 0:
        print(f"Embedded {i}/{len(admission_drugs)} admissions")

# ------------------------------
# SAVE OUTPUTS
# ------------------------------
if admission_embeddings:
    X = np.vstack(admission_embeddings)
    hadm_ids = np.array(admission_ids)
    subject_ids = np.array(subject_ids)

    np.save(OUTPUT_PATH, X)
    np.save(OUTPUT_PATH.replace(".npy","_hadm_ids.npy"), hadm_ids)
    np.save(OUTPUT_PATH.replace(".npy","_subject_ids.npy"), subject_ids)

    print(f"\nSaved {len(hadm_ids)} admission embeddings")
    print("Embedding shape:", X.shape)
else:
    print("No embeddings computed. Check your bridge and XML mapping.")


Using device: cuda


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

MedGemma model loaded.

Found 21769 cancer admissions.

Filtered prescriptions to 977852 rows for target cancer admissions.

Parsing DrugBank XML...
Total drugs with descriptions parsed: 12439

Total admissions with drug descriptions: 21158

Embedded 50/21158 admissions
Embedded 100/21158 admissions
Embedded 150/21158 admissions
Embedded 200/21158 admissions
Embedded 250/21158 admissions
Embedded 300/21158 admissions
Embedded 350/21158 admissions
Embedded 400/21158 admissions
Embedded 450/21158 admissions
Embedded 500/21158 admissions
Embedded 550/21158 admissions
Embedded 600/21158 admissions
Embedded 650/21158 admissions
Embedded 700/21158 admissions
Embedded 750/21158 admissions
Embedded 800/21158 admissions
Embedded 850/21158 admissions
Embedded 900/21158 admissions
Embedded 950/21158 admissions
Embedded 1000/21158 admissions
Embedded 1050/21158 admissions
Embedded 1100/21158 admissions
Embedded 1150/21158 admissions
Embedded 1200/21158 admissions
Embedded 1250/21158 admissions
Emb